提示词（Prompt）就是发送给模型的消息，其中SystemMessage是系统提示词，可以给模型设定角色、聊天背景、任务说明，对模型生成的内容有很大影响

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

创建智能体时，就可以直接指定系统提示词

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model="deepseek-v4-flash"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage("你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model="deepseek-v4-flash",
    system_prompt="你以海盗的口吻来回答用户问题。"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage("你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

哟，问得好！老子就是七大洋上最臭名昭著的“血红约翰”——加勒比海最让人闻风丧胆的海盗船长！你要是身上没带金币，我建议你赶紧跑，否则我就把你扔去喂鲨鱼！

## 2.提示词工程
所谓提示词（Prompt Engineering），就是通过优化提示词使模型输出的结果更符合业务需要的过程

系统提示词会包含以下几个部分，通常按此顺序排列：
- 身份角色：描述AI的职责、沟通风格和总体目标
- 指令说明：指导模型如何生成所需的响应，应该遵循哪些规则，模型应该做什么，以及模型绝对不能做什么
- 对话示例：提供可能的输入示例，以及模型期望的输出
- 背景信息：向模型提供生成响应所需的任何额外信息，如RAG的额外知识库数据，或特别相关的任何其他数据  

编写System Prompt时，可以使用Markdown格式和XML标签的组合来帮助模型理解提示和上下文数据的逻辑边界

### 2.1.设定角色和指令

In [ ]:
system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = 'deepseek-v4-pro',
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎么定义string变量记录学校名字")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

添加指令描述，进一步约束模型的行为，什么能做，什么不能做

In [ ]:
system_prompt = """
# 身份
你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可
"""

# 创建智能体
agent = create_agent(
    model = 'deepseek-v4-pro',
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎么定义string变量记录学校名字")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

### 2.2.对话示例（example）

In [3]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = 'deepseek-v4-pro',
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

目前金星没有已知的首都，因为这颗行星表面环境极端恶劣——高温、高压和硫酸云让人类无法居住。

但如果从科幻角度，我倒可以为你构思一个**太空时代的金星首都**——

比如在金星大气层中的浮空城市群，首都叫「**磷光城**」，悬浮在离金星表面约50公里高处——那一层的气温和气压接近地球，城市漂浮在浓厚的二氧化碳大气中，靠巨大的气囊和反重力装置维持高度；外壁能抵御硫酸云，内部街道却布满人造阳光和垂直森林，是太阳系内环最繁华的贸易港之一。

你是想听基于现实的金星数据，还是更想探索这种未来设定？

In [ ]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创造一个太空之都。
# 示例
user:
assistant: 
"""

# 创建智能体
agent = create_agent(
    model = 'deepseek-v4-pro',
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎么定义string变量记录学校名字")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

### 2.3.结构化输出

a.基于提示词的结构化输出

In [ ]:
system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创造一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例
user: 月球的首都都是什么？
assistant: 
{
    "name": "月华市(Lunaria)",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采】量子通信枢纽、尖端生物圈农业"
}
"""

# 创建智能体
agent = create_agent(
    model = 'deepseek-v4-pro',
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都都是什么？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

{
    "name": "辉云城(Nephopolis)",
    "location": "位于金星北半球伊师塔地高原的云端，悬浮在离地表约50公里的宜居大气层中",
    "vibe": "轻盈、梦幻、仿生",
    "economy": "太阳能风暴发电、碳纤维复合材料生产、高酸性环境生物科技"
}

#### b.基于Model的结构化输出  
在LangChain中，实现结构化输出会更加简单，无需自己在提示词中添加描述实现结构化输出，仅仅是两步即可：  
- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式

In [6]:
from pydantic import BaseModel

# 定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [8]:
# 创建智能体时设置结构化输出的格式，LangChain会自动完成提示词改造和响应结果解析
agent = create_agent(
    model="deepseek-chat",
    system_prompt="你是一个科幻作家，根据用户的要求创造一个太空之都。",
    response_format=CapitalInfo  # 将响应消息设置为结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都都是什么？")]}
)

In [9]:
city = response['structured_response']
city

CapitalInfo(name='月海之都（Lunar Capital）', location='月球静海（Mare Tranquillitatis）地下城市群', vibe='赛博朋克与古典文明交织，穹顶之下是永恒的人造黄昏', economy='氦-3能源出口、低重力制造业、星际旅游枢纽')